In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import f1_score
import yfinance as yf
from pprint import pprint

# Data Import

In [10]:
df = yf.download("AAPL", start="2020-01-01", end="2024-12-31")
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
df.tail()

[*********************100%***********************]  1 of 1 completed


Price,Open,High,Low,Close,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2024-12-23,253.385834,254.261043,252.072998,253.883118,40858800
2024-12-24,254.101927,256.807136,253.903002,256.797211,23234700
2024-12-26,256.787255,258.686881,256.230300,257.612732,27237100
2024-12-27,256.429206,257.294505,251.685132,254.201385,42355300
2024-12-30,250.859639,252.122744,249.387684,250.829803,35557500


# Questions

## 1.
Create new features:
- `lag_1`: 1-day lag of the opening price.
- `ma_5`: 5-day moving average of the opening price.
- `ma_10`: 10-day moving average of the opening price.
- `ma_20`: 20-day moving average of the opening price.
- `sd_5`: 5-day standard deviation of the opening price.
- `sd_10`: 10-day standard deviation of the opening price.
- `sd_20`: 20-day standard deviation of the opening price.

In [11]:
df_model = pd.DataFrame(
    {
        "lag_1": df['Open'].shift(1).values[:, 0],
        "ma_5": df['Open'].rolling(window=5).mean().values[:, 0],
        "ma_10": df['Open'].rolling(window=10).mean().values[:, 0],
        "ma_20": df['Open'].rolling(window=20).mean().values[:, 0],
        "sd_5": df['Open'].rolling(window=5).std().values[:, 0],
        "sd_10": df['Open'].rolling(window=10).std().values[:, 0],
        "sd_20": df['Open'].rolling(window=20).std().values[:, 0],
    },
    index=df.index
)
df_model["target"] = df["Open"].values[:, 0] - df_model["lag_1"]
df_model["target"] = df_model["target"].apply(lambda x: 1 if x > 0 else -1)
df_model.dropna(inplace=True)
df_model

,lag_1,ma_5,ma_10,ma_20,sd_5,sd_10,sd_20,target
Date,,,,,,,,
2020-01-30,78.209915,76.553874,76.449016,74.891325,1.448520,1.020967,2.216845,-1
2020-01-31,77.267393,76.586659,76.625951,75.188906,1.468446,1.009200,2.122345,1
2020-02-03,77.361405,76.308966,76.337410,75.275083,1.955278,1.449047,2.001761,-1
2020-02-04,73.352688,76.439615,76.292090,75.534456,1.896406,1.451875,1.708629,1
2020-02-05,76.006676,76.394780,76.411171,75.819865,1.846071,1.543635,1.609859,1
...,...,...,...,...,...,...,...,...
2024-12-23,246.692406,249.148975,247.657122,242.318787,2.992637,2.568257,6.553102,1
2024-12-24,253.385834,250.225099,248.512450,243.513759,3.687212,3.147089,6.404761,1
2024-12-26,254.101927,251.424550,249.529893,244.750006,4.741611,3.995127,6.463999,1


## 2.
Split the dataset into training and testing sets, using the first 70% of the data for training, 20% for validation, and the remaining 10% for testing.

In [12]:
df_train, df_test = train_test_split(df_model, test_size=0.3, shuffle=False)
df_val, df_test = train_test_split(df_test, test_size=0.33, shuffle=False)
df_model.shape, df_train.shape, df_val.shape, df_test.shape

((1238, 8), (866, 8), (249, 8), (123, 8))

## 3.

### 3.1
Train a random forest classifier model to predict whether the opening price will increase or decrease the next day, using the created features as input. Tune the hyperparameters of the model:
- `n_estimators`
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`
- `max_features`

:::{.callout-tip}
Use the `RandomForestRegressor` class from the `sklearn.ensemble` module to train the model, the `GridSearchCV` class from the `sklearn.model_selection` module for cross-validation.
:::

In [13]:
hyperparams = {
    'n_estimators': [10, 30, 50],
    'max_depth': [1, 3, 5],
    'min_samples_leaf': [1, 5, 10],
    'min_samples_split': [10, 20, 50],
    'max_features': [5, 'sqrt', 'log2']
}

In [14]:
idx_train = [i for i in range(df_train.shape[0])]
idx_val = [i for i in range(df_train.shape[0], df_train.shape[0] + df_val.shape[0])]
idx_train[-1], idx_val[0], idx_val[-1]

(865, 866, 1114)

In [15]:
df_cv = pd.concat([df_train, df_val], axis=0)
df_cv.shape

(1115, 8)

In [16]:
cv = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=hyperparams,
    cv=[(idx_train, idx_val)],
    scoring='f1',
)
cv

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [1, 3, ...], 'max_features': [5, 'sqrt', ...], 'min_samples_leaf': [1, 5, ...], 'min_samples_split': [10, 20, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[([0, 1, ...], ...)]"
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time

In [17]:
cv.fit(df_cv.drop('target', axis=1), df_cv['target'])

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [1, 3, ...], 'max_features': [5, 'sqrt', ...], 'min_samples_leaf': [1, 5, ...], 'min_samples_split': [10, 20, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[([0, 1, ...], ...)]"
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time

In [22]:
pd.DataFrame(cv.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split,param_n_estimators,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
0,0.008988,0.0,0.001808,0.0,1,5,1,10,10,"{'max_depth': 1, 'max_features': 5, 'min_sampl...",0.501931,0.501931,0.0,40
1,0.015214,0.0,0.001764,0.0,1,5,1,10,30,"{'max_depth': 1, 'max_features': 5, 'min_sampl...",0.126761,0.126761,0.0,196
2,0.023497,0.0,0.002483,0.0,1,5,1,10,50,"{'max_depth': 1, 'max_features': 5, 'min_sampl...",0.089552,0.089552,0.0,204
3,0.005251,0.0,0.001170,0.0,1,5,1,20,10,"{'max_depth': 1, 'max_features': 5, 'min_sampl...",0.501931,0.501931,0.0,40
4,0.014273,0.0,0.001550,0.0,1,5,1,20,30,"{'max_depth': 1, 'max_features': 5, 'min_sampl...",0.578073,0.578073,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,0.017622,0.0,0.001639,0.0,5,log2,10,20,30,"{'max_depth': 5, 'max_features': 'log2', 'min_...",0.442553,0.442553,0.0,76
239,0.028854,0.0,0.002265,0.0,5,log2,10,20,50,"{'max_depth': 5, 'max_features': 'log2', 'min_...",0.085714,0.085714,0.0,206
240,0.006264,0.0,0.001201,0.0,5,log2,10,50,10,"{'max_depth': 5, 'max_features': 'log2', 'min_...",0.376238,0.376238,0.0,115
241,0.017351,0.0,0.001661,0.0,5,log2,10,50,30,"{'max_depth': 5, 'max_features': 'log2', 'min_...",0.242424,0.242424,0.0,165


In [27]:
cv.best_params_

{'max_depth': 1,
 'max_features': 'sqrt',
 'min_samples_leaf': 5,
 'min_samples_split': 50,
 'n_estimators': 10}

### 3.2
Evaluate the performance of the model using appropriate metrics.

In [23]:
f1_score(df_test['target'], cv.predict(df_test.drop('target', axis=1)))

0.5780346820809249

## 4.

### 4.1.
Train a gradient boosting tree classifier model to predict whether the opening price will increase or decrease the next day, using the created features as input. Tune the hyperparameters of the model:
- `n_estimators`
- `learning_rate`
- `max_depth`
- `subsample`

:::{.callout-tip}
Use the 'GradientBoostingClassifier' class from the 'sklearn.ensemble' module to train a gradient boosting classifier model, and compare its performance with the random forest model.
:::

### 4.2
Evaluate the performance of the model using appropriate metrics.